### SUMMARY
- Number of simulations processed: 256
- Min simulated time: 1.04 ms
- Max simulated time: 2.11 ms

### ESTIMATED COMPUTATIONAL RESOURCES:
- Number of restarts: 21
- Total node-hours: 16.9 knode-hours
- Total GPUhs: 67.5 kGPU-hours
- Total wall clock time: 5.4 Days

### DATASET
- ~80 TB of data generated

(run `check_computational_cost.py` to reproduce)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import h5py

from scan_analysis.scan_metadata import ScanMetadata
tmp_figdir = '/Users/ahoffman/personal_gkyl_scripts/studies/tcv_miller_scan_scripts/figures/'
manuscript_figdir = '/Users/ahoffman/writing/gkyl_tcv_miller_scan/figures/'
# h5file = 'data/tcv_miller_scan_big_metadata_frame_500_navg_25_lingk_63.h5'
h5file = 'data/tcv_miller_scan_big_metadata_lastframes_navg_50_esgk.h5'
miller_scan = ScanMetadata(h5file)
h5file = 'data/tcv_miller_scan_big_metadata_lastframes_navg_50_emgk.h5'
miller_scan_em = ScanMetadata(h5file)
miller_scan.info()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
kappa = 1.1
delta = 0.6
energ = 5e6
idx_pt = miller_scan.get_sim_index({'kappa':kappa,'delta':delta, 'energy_srcCORE':energ})
idx_nt = miller_scan.get_sim_index({'kappa':kappa,'delta':-delta, 'energy_srcCORE':energ})
ky_pt = miller_scan.metadata[idx_pt]['linear_gk']['ky']
omega_pt = miller_scan.metadata[idx_pt]['linear_gk']['omega']
ky_nt = miller_scan.metadata[idx_nt]['linear_gk']['ky']
omega_nt = miller_scan.metadata[idx_nt]['linear_gk']['omega']

fig,ax = plt.subplots(1,1,figsize=(5,3.5))
ax.plot(ky_pt,omega_pt.real,'-',color='r')
ax.plot(ky_nt,omega_nt.real,'-',color='b')
ax.plot(ky_pt,omega_pt.imag,'--',color='r')
ax.plot(ky_nt,omega_nt.imag,'--',color='b')
ax.set_xlabel(r'$k_y \rho_s$')
ax.set_ylabel(r'$\gamma R/c_s, \omega R/c_s$')


fig.tight_layout()

In [ ]:
figdir = manuscript_figdir
miller_scan.plot_field_vs_field('omega', 'gamma', ky=0.5, alpha=0.5, xlim=[-10,-3.8],ylim=[0,6.5],
                                figfilename=figdir+'gk_lin_omega_vs_gamma_lcfs.png',dpi=300, edgecolor='gray',
                                lines =[])

In [ ]:
miller_scan.plot_field_vs_field('kTi', 'gamma', ky=0.5, alpha=0.5)

In [ ]:
miller_scan.plot_field_vs_field('kTi', 'omega', ky=0.5, alpha=0.5)

In [ ]:
ky = 0.2
miller_scan_em.plot_field_vs_field('betae_lcfs', 'gamma',
                                xlim=[0,0.08], ylim=[0,5],
                                ky=ky, alpha=0.5)
miller_scan_em.plot_field_vs_field('betae_lcfs', 'omega',
                                xlim=[0,0.08], ylim=[-15,15],
                                ky=ky, alpha=0.5)
miller_scan_em.plot_field_vs_field('omega', 'gamma',
                                xlim=[-15,15], ylim=[0,5],
                                ky=ky, alpha=0.5)

In [ ]:
with h5py.File(h5file, 'r') as f:
    
    fig, axs = plt.subplots(2,2)
    axs = axs.flatten()
    scan_table = [key for key in f.keys()]
    colors = ['C0', 'C1', 'C2', 'C3']
    for n in range(0, len(scan_table), 4):
        ic = 0
        for k in range(4):
            ax = axs[k]
            for key in scan_table[n+k:n+k+1]:
                y = 0
                y += f[key]['intmom']['bflux_x_u_He']['values'][:]
                y += f[key]['intmom']['bflux_z_u_He']['values'][:]
                y += f[key]['intmom']['bflux_z_l_He']['values'][:]
                y += f[key]['intmom']['bflux_x_u_Hi']['values'][:]
                y += f[key]['intmom']['bflux_z_u_Hi']['values'][:]
                y += f[key]['intmom']['bflux_z_l_Hi']['values'][:]   
                
                y = y/y[0]
                
                t = f[key]['intmom']['bflux_x_u_He']['time'][:]/1000
            
                ax.plot(t, y, label=key, alpha=0.1, color=colors[ic])
                ic += 1
            ax.set_xlabel('ms')
            ax.set_ylabel('MW')
            ax.set_ylim([0,2.0])
fig.tight_layout()

In [ ]:
figdir = tmp_figdir

with h5py.File(h5file, 'r') as f:
    
    fig, axs = plt.subplots(2,2)
    axs = axs.flatten()
    scan_table = [key for key in f.keys()]
    colors = ['C0', 'C1', 'C2', 'C3']
    for n in range(0, len(scan_table), 4):
        ic = 0
        for k in range(4):
            for key in scan_table[n+k:n+k+1]:
                y = 0
                y += f[key]['intmom']['We']['values'][:] 
                y += f[key]['intmom']['Wi']['values'][:] 
                t = f[key]['intmom']['We']['time'][:]/1000
                
                y = np.gradient(y, t)*1000
                
                # smooth y with a moving average
                window_size = 25
                y = np.convolve(y, np.ones(window_size)/window_size, mode='valid')
                t = t[window_size-1:]
            
                axs[k].plot(t, y, label=key, alpha=0.1, color=colors[ic])
                ic += 1
            # ax.set_xlim([0,1])
            axs[k].set_ylim([-1.0,1.0])

for ax in axs:
    ax.set_xlabel(r'$t$ [ms]')
    ax.set_ylabel(r'$dW/dt$ [MW]')
fig.tight_layout()
fig.savefig(figdir+'energy_time_traces.png', dpi=300)

In [ ]:
with h5py.File(h5file, 'r') as f:
    
    fig, axs = plt.subplots(2,2)
    axs = axs.flatten()
    scan_table = [key for key in f.keys()]
    colors = ['C0', 'C1', 'C2', 'C3']
    for n in range(0, len(scan_table), 4):
        ic = 0
        for k in range(4):
            ax = axs[k]
            for key in scan_table[n+k:n+k+1]:
                y = 0
                y += f[key]['intmom']['ne']['values'][:] 
                y = y/y[0]
                t = f[key]['intmom']['ne']['time'][:]/1000
            
                ax.plot(t, y, label=key, alpha=0.1, color=colors[ic])
                ic += 1
fig.tight_layout()

In [ ]:
figdir = tmp_figdir

with h5py.File(h5file, 'r') as f:
    
    fig, ax = plt.subplots(1,1, figsize=(5,3.5))
    scan_table = [key for key in f.keys()]
    colors = ['C0', 'C1', 'C2', 'C3']
    for n in range(0, len(scan_table), 4):
        ic = 0
        for k in range(4):
            for key in scan_table[n+k:n+k+1]:
                y = 0
                y += f[key]['intmom']['We']['values'][:] 
                y += f[key]['intmom']['Wi']['values'][:] 
                t = f[key]['intmom']['We']['time'][:]/1000
                
                y = y/y[0]
            
                ax.plot(t, y, label=key, alpha=0.25, color=colors[ic])
                ic += 1
            # ax.set_xlim([0,1])
            ax.set_ylim([0,2.0])

ax.set_xlabel(r'$t$ [ms]')
ax.set_ylabel(r'$W/W_0$')
fig.tight_layout()
fig.savefig(figdir+'energy_time_traces.png', dpi=300)

In [ ]:
miller_scan.plot_field_vs_field('energy_srcCORE', 'bflux_wall_H', alpha=0.5)


In [ ]:
lines = []
lines.append({'x': [10,100], 'y': [30,300], 'kwargs': {'color': 'k', 'linestyle':'dashed'}})

miller_scan.plot_field_vs_field('Te_limup', 'phi_limup', alpha=0.5, lines=lines)
miller_scan.plot_field_vs_field('Te_limlo', 'phi_limlo', alpha=0.5, lines=lines)

In [ ]:
lines = []
lines.append({'x': [0,1], 'y': [0,1], 'kwargs': {'color': 'k', 'linestyle':'dashed'}})
lines.append({'x': [0,1], 'y': [0,1.5], 'kwargs': {'color': 'k', 'linestyle':'dashed'}})
miller_scan.plot_field_vs_field('bflux_z_l_H', 'bflux_z_u_H', alpha=0.5, xlim=[0.1,0.55], ylim=[0.1,0.55], lines=lines)

In [ ]:
lines = []
lines.append({'x': [0,1], 'y': [0.1,3.3], 'kwargs': {'color': 'k', 'linestyle':'dashed'}})
miller_scan.plot_field_vs_field('bflux_x_u_H', 'bflux_z_u_H', alpha=0.5, lines=lines)

In [ ]:
miller_scan.plot_field_vs_field('bflux_z_u_H', 'lambda_q', alpha=0.5, lines=[])

In [ ]:
miller_scan.plot_field_vs_field('Pi_norm_limup', 'Pi_norm_edge', alpha=0.5, lines=[])
miller_scan.plot_field_vs_field('ne_norm_limup', 'ne_norm_edge', alpha=0.5, lines=[])
miller_scan.plot_field_vs_field('Ti_norm_limup', 'Ti_norm_edge', alpha=0.5, lines=[])
miller_scan.plot_field_vs_field('Te_norm_limup', 'Te_norm_edge', alpha=0.5, lines=[])

In [ ]:
figdir = tmp_figdir
miller_scan.plot_field_vs_field('Ti_norm_limup', 'Ti_norm_edge', powers=[1e5,5e5, 1e6, 5e6], alpha=0.5, 
                                figfilename=figdir+'Ti_limup_vs_Ti_edge.png',dpi=300, edgecolor='gray')

In [ ]:
figdir = manuscript_figdir
miller_scan.plot_field_vs_field('Ti_norm_sol', 'Ti_norm_edge', 
                                xlim = [0.8,1.2],
                                ylim = [1.06, 1.35],
                                powers=[1e5,5e5, 1e6, 5e6], 
                                alpha=0.5, 
                                figfilename=figdir+'Ti_sol_vs_Ti_edge.png',dpi=300,
                                edgecolor='gray')

In [ ]:
figdir = manuscript_figdir
miller_scan.plot_field_vs_field('Te_norm_sol', 'Te_norm_edge', 
                                xlim = [0.25,0.6],
                                ylim = [1.4, 2.2],
                                powers=[1e5,5e5, 1e6, 5e6], 
                                alpha=0.5, 
                                figfilename=figdir+'Te_sol_vs_Te_edge.png',dpi=300,
                                edgecolor = 'gray')

In [ ]:
figdir = ''
miller_scan.plot_field_vs_field('tau_sol', 'tau_edge', 
                                xlim = [1.5, 5.0],
                                ylim = [0.6, 1.4],
                                powers=[1e5,5e5, 1e6, 5e6], 
                                alpha=0.5, 
                                figfilename=figdir+'tau_sol_vs_tau_edge.png',dpi=300,
                                edgecolor = 'gray')

In [ ]:
figdir = manuscript_figdir
miller_scan.plot_field_vs_field('kne', 'kTi', 
                                xlim = [21.5,29.5],
                                ylim = [2, 10],
                                powers=[1e5,5e5, 1e6, 5e6], 
                                alpha=0.5, 
                                figfilename=figdir+'kn_vs_kTi_edge.png',dpi=300,
                                edgecolor = 'gray')

In [ ]:
figdir = tmp_figdir
miller_scan.plot_field_vs_field('kTe', 'kTi', 
                                xlim = [11.5,19.5],
                                ylim = [2, 10],
                                powers=[1e5,5e5, 1e6, 5e6], 
                                alpha=0.5, 
                                edgecolor = 'gray',
                                figfilename=figdir+'kn_vs_kTi_edge.png',
                                dpi=300)

In [ ]:
figdir = manuscript_figdir

itg_tem_region = {'x': [1/4,1], 'y': [-1/10 - 1/3,-1/10 + 1/3], 'kwargs': {'color': 'y', 'alpha': 0.1, 'linestyle':'dashed'}}
lines = []
shadows = [itg_tem_region]
miller_scan.plot_field_vs_field('chixe_over_chixi', 'De_over_chitot', powers=[1e5, 5e5, 1e6, 5e6], 
                                alpha=0.5, 
                                edgecolor = 'gray',
                                figfilename=figdir+'instability_classification.png',dpi=300,
                                xlim = [0.1, 1.2], ylim=[-0.5,0.3], 
                                lines=lines, shadows=shadows)

In [ ]:
miller_scan.plot_field_vs_field('pflux_xe_edge', 'hflux_xe_edge', alpha=0.5, lines=[])
miller_scan.plot_field_vs_field('pflux_xi_edge', 'hflux_xi_edge', alpha=0.5, lines=[])

In [ ]:
figdir = tmp_figdir
miller_scan.plot_field_vs_field('delta', 'lambda_q', 
                                powers=[1e5,5e5, 1e6, 5e6], 
                                ylim = [0.001,0.01],
                                alpha=0.5, 
                                figfilename=figdir+'delta_vs_lambda_q.png',
                                dpi=300)

In [ ]:
figdir = tmp_figdir

power = 0.1
fixed_param = {'energy_srcCORE': power*1e6}
method = 'contourf'  # Use pcolormesh for better handling of log-scale axes
miller_scan.plot_contour_grid(['avg_dt'], fixed_params=fixed_param, method=method, figfilename=f'{figdir}avg_dt_contour_grid_power_{power}MW.png')

In [ ]:
power = 5e6
miller_scan.plot_contour_grid(['chixe_over_chixi'], fixed_params={'energy_srcCORE': power}, method='pcolormesh',
                              cmap='coolwarm', clim=[0.1,10])
miller_scan.plot_contour_grid(['De_over_chitot'], fixed_params={'energy_srcCORE': power}, method='pcolormesh',
                              cmap='coolwarm', clim=[-1/10-1/3,1/10+1/3])

In [ ]:
from scan_analysis import find_bounce_angle

# Draw a poloidal cut of a flux surface given kappa and delta
# --- User Parameters ---
Raxis = 0.8727315068
Rlcfs = 1.0968432365089495
Rmin = Rlcfs - 0.04
r0 = Rlcfs - Raxis
kappa = 1.4
delta = 0.45
q_val = 2.6

theta = np.linspace(-np.pi, np.pi, 100)
R = lambda r: Raxis + r*np.cos(theta + np.arcsin(delta)*np.sin(theta))
Z = lambda r: r*kappa*np.sin(theta)

# Create a banana orbit trajectory
r_ban = 0.025  # the banana orbit width
theta_ban = np.pi/3
theta_ban = find_bounce_angle(Rb = 0.9, r=r0, delta=delta, R0=Raxis)

# Poloidal angle range spanning both tips
theta_orbit = np.linspace(-theta_ban, theta_ban, 200)

# Orbit half-width: zero at the tips (theta = ±theta_ban), maximum r_ban at the outboard (theta = 0)
# Derived from the trapping condition: delta_r ~ sqrt(cos(theta) - cos(theta_ban))
delta_r = r_ban * np.sqrt(
    np.maximum(0, (np.cos(theta_orbit) - np.cos(theta_ban)) / (1 - np.cos(theta_ban)))
)

# Outer leg: particle drifting at r = Raxis + delta_r (outward on outboard side)
R_ban_outer = Raxis + (r0 + delta_r) * np.cos(theta_orbit + np.arcsin(delta) * np.sin(theta_orbit))
Z_ban_outer = kappa * (r0 + delta_r) * np.sin(theta_orbit)

# Inner leg: return trajectory at r = Raxis - delta_r (traversed in reverse to close the orbit)
R_ban_inner = Raxis + (r0 - delta_r[::-1]) * np.cos(theta_orbit[::-1] + np.arcsin(delta) * np.sin(theta_orbit[::-1]))
Z_ban_inner = kappa * (r0 - delta_r[::-1]) * np.sin(theta_orbit[::-1])

R_ban = np.concatenate([R_ban_outer, R_ban_inner])
Z_ban = np.concatenate([Z_ban_outer, Z_ban_inner])



fig, ax = plt.subplots(figsize=(5,3.5))
ax.plot(R(r0), Z(r0), label='flux surface', color='C0')
ax.plot(R(0.9*r0), Z(0.9*r0), label='flux surface', color='C0')
ax.plot(R(0.8*r0), Z(0.8*r0), label='flux surface', color='C0')
ax.plot(R_ban, Z_ban, label='banana orbit', color='r')
## Create a shaded area at the high field side of the flux surface
ax.add_patch(plt.Rectangle((2*Raxis-Rlcfs-0.02, -0.2), 0.08, 0.4, color='b', alpha=0.1, label='high-field side'))
## Add the limiter
ax.add_patch(plt.Rectangle((2*Raxis-Rlcfs-0.08, -0.005), 0.08, 0.01, color='k', alpha=1.0, label='limiter'))
ax.axis('equal')
# remove the axis
# ax.set_xticks([])
# ax.set_yticks([])
# remove the box
ax.set_frame_on(False)
fig.tight_layout()

In [ ]:
from scan_analysis import calculate_arc_length, get_theta_from_R
delta=0.45
# --- Compute & Plot ---
theta_pt_plus, l_pt_plus = calculate_arc_length(r0, kappa, delta, q_val, R0=Raxis, theta_min=0, theta_max=np.pi)
theta_nt_plus, l_nt_plus = calculate_arc_length(r0, kappa, -delta, q_val, R0=Raxis, theta_min=0, theta_max=np.pi)

theta_nt_minus, l_nt_minus = calculate_arc_length(r0, kappa, -delta, q_val, R0=Raxis, theta_min=0, theta_max=-np.pi)
theta_pt_minus, l_pt_minus = calculate_arc_length(r0, kappa, delta, q_val, R0=Raxis, theta_min=0, theta_max=-np.pi)

theta_nt = np.concatenate([theta_nt_minus, theta_nt_plus])
l_nt = np.concatenate([l_nt_minus, l_nt_plus])
theta_pt = np.concatenate([theta_pt_minus, theta_pt_plus])
l_pt = np.concatenate([l_pt_minus, l_pt_plus])

# sort in increasing theta
nt_sort_idx = np.argsort(theta_nt)
theta_nt = theta_nt[nt_sort_idx]
l_nt = l_nt[nt_sort_idx]
pt_sort_idx = np.argsort(theta_pt)
theta_pt = theta_pt[pt_sort_idx]
l_pt = l_pt[pt_sort_idx]

th_bounce_pt = find_bounce_angle(Rb = 0.9, r=r0, delta=delta, R0=Raxis)
th_bounce_nt = find_bounce_angle(Rb = 0.9, r=r0, delta=-delta, R0=Raxis)

recy_center = 1.0
recy_sigma = 2 * 0.05

# in function of R instead of theta

R_th_pt = lambda th: Raxis + r0 * np.cos(th + np.arcsin(delta) * np.sin(th))
R_th_nt = lambda th: Raxis + r0 * np.cos(th + np.arcsin(-delta) * np.sin(th))

fig,ax = plt.subplots(figsize=(5,3.5))
d_nt = np.minimum(np.abs(l_nt-l_nt[-1]),np.abs(l_nt-l_nt[0]))
ax.plot(R_th_nt(theta_nt), d_nt, label='NT', color='C0')

d_pt = np.minimum(np.abs(l_pt-l_pt[-1]),np.abs(l_pt-l_pt[0]))
ax.plot(R_th_pt(theta_pt), d_pt, label='PT', color='C1')
ax.axvline(R_th_pt(th_bounce_pt), color='k', linestyle='dashed', label='PT bounce point')

# Add a color gradient for the recycling region
sigma_z = 2 * 0.05*2*np.pi
_,dsigma_nt = calculate_arc_length(r0, kappa, -delta, q_val, theta_min=-np.pi, theta_max=-np.pi+sigma_z, R0=Raxis)
_,dsigma_pt = calculate_arc_length(r0, kappa, delta, q_val, theta_min=-np.pi, theta_max=-np.pi+sigma_z, R0=Raxis)
ax.axhspan(0, dsigma_nt[-1], color='C0', alpha=0.2, label='recycling region')
ax.axhspan(0, dsigma_pt[-1], color='C1', alpha=0.2, label='recycling region')

ax.set_xlabel(r'$R$ [m]')
ax.set_ylabel(r'$d_\ell$ [m]')
ax.set_xlim([2*Raxis-Rlcfs-0.08, Rlcfs+0.02])
ax.set_ylim([0, 8.0])
ax.grid(False)




In [ ]:
ncall = 352006
time = 1.9274e+04
time_per_call = time/ncall
print(f"CSBC Time per call: {time_per_call*1000:.4f} ms")

ncall = 341665
time = 1.9280e+04
time_per_call = time/ncall
print(f"AIBC Time per call: {time_per_call*1000:.4f} ms")

In [ ]:
from scan_analysis import calculate_arc_length, get_theta_from_R
Rb = 0.9

delta_vals = np.linspace(0.001,0.9, 32)

dl_Rb = np.zeros_like(delta_vals)

for i, delta in enumerate(delta_vals):
    dl_delta = 0
    for d_ in [delta, -delta]:
        theta_plus, dl_plus = calculate_arc_length(r0, kappa, d_, q_val, R0=Raxis, theta_min=0, theta_max=np.pi)
        theta_minus, dl_minus = calculate_arc_length(r0, kappa, d_, q_val, R0=Raxis, theta_min=0, theta_max=-np.pi)
        theta = np.concatenate([theta_minus, theta_plus])
        l = np.concatenate([dl_minus, dl_plus])
        # sort in increasing theta
        sort_idx = np.argsort(theta)
        theta = theta[sort_idx]
        l = l[sort_idx]
        dl = np.minimum(np.abs(l-l[-1]),np.abs(l-l[0]))
    
        theta_b = find_bounce_angle(Rb = Rb, r=r0, delta=d_, R0=Raxis)
        dl_Rb_ = np.interp(theta_b, theta, dl)
        
        dl_delta += dl_Rb_ if d_ > 0 else -dl_Rb_
    dl_Rb[i] = dl_delta
    
fig, ax = plt.subplots(figsize=(5,3.5))
ax.plot(delta_vals, dl_Rb)
ax.set_xlabel(r'$|\delta|$')
ax.set_ylabel(r'$d_\ell(\delta)-d_\ell(-\delta)$')
ax.set_title(r'$R_b=$'+f'{Rb} m,'+r' $\kappa=$'+f'{kappa}, '+r' $q=$'+f'{q_val}')
ax.grid(False)
ax.set_xlim([0, None])
ax.set_ylim([0, None])
plt.tight_layout()
plt.show()


